In [0]:
from typing import Any

import requests


class APIReader:


    def __init__(self, base_url: str,timeout: int = 30,default_headers: dict | None = None) -> None:
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update(
            {"Accept": "application/json", **(default_headers or {})}
        )

    def _url(self, endpoint: str) -> str:
        return f"{self.base_url}/{endpoint.lstrip('/')}"

    def _request(self, method: str, endpoint: str, **kwargs) -> Any:
        kwargs.setdefault("timeout", self.timeout)
        response = self.session.request(method, self._url(endpoint), **kwargs)
        response.raise_for_status()

        if not response.content:
            return None
        if "application/json" in response.headers.get("Content-Type", ""):
            return response.json()
        return response.text

    def get(self, endpoint: str, params: dict | None = None, **kwargs) -> Any:
        return self._request("GET", endpoint, params=params, **kwargs)

    def post(self, endpoint: str, json: dict | None = None, **kwargs) -> Any:
        return self._request("POST", endpoint, json=json, **kwargs)

    def put(self, endpoint: str, json: dict | None = None, **kwargs) -> Any:
        return self._request("PUT", endpoint, json=json, **kwargs)

    def delete(self, endpoint: str, **kwargs) -> Any:
        return self._request("DELETE", endpoint, **kwargs)

    def close(self) -> None:
        self.session.close()

    def __enter__(self) -> "SimpleAPIReader":
        return self

    def __exit__(self, *exc) -> None:
        self.close()

    def __repr__(self) -> str:
        return f"SimpleAPIReader(base_url='{self.base_url}')"

In [0]:
CEP = "60055210"
api = APIReader("https://viacep.com.br")
dados = api.get(f"ws/{CEP}/json/")

df = spark.createDataFrame([dados])
df.display()

In [0]:
print(dados)